In [2]:
import yaml
import numpy as np 
from pytorch_lightning import Trainer
import importlib

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys 
sys.path.append('../')
from src import cv_word_lightning
importlib.reload(cv_word_lightning)

CommonVoiceWordRec = cv_word_lightning.CommonVoiceWordRec

In [4]:
path = 'config/commonvoice/cv_word_baseline.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [5]:
config

{'corpus': {'root': '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/'},
 'audio': {'rep_type': 'cochlea_filt',
  'rep_kwargs': {'sr': 50000,
   'env_sr': 10000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'rep_on_gpu': True,
   'center_crop': True,
   'out_dur': 2,
   'impulse_len': 0.25,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'clip_value': 5,
   'power': 0.3}},
 'loader': {'batch_size': 256},
 'model': {'num_words': 998, 'fc_size': 512},
 'val_metric': 'ACC/val_acc',
 'model_name': 'cv_clean_word_rec_baseline',
 'hparas': {'valid_step': 500,
  'epochs': 10,
  'optimizer': 'Adam',
  'lr': 0.001,
  'eps': 1e-08,
  'lr_

In [4]:
!nvidia-smi

No devices were found


In [13]:
config['loader']['num_workers'] = 0
config['loader']['batch_size'] = 1

config['audio']['rep_kwargs']['rep_on_gpu'] = False

In [14]:
model = CommonVoiceWordRec(config)

using IIR cochleagram


In [15]:
import pickle 
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_word_int_label_dict.pkl", "rb" )) 


In [16]:
import h5py

In [17]:
len(class_map)

998

In [22]:
trainer = Trainer(
    precision=32,
    default_root_dir='test_log_dump/',
    val_check_interval=1,
#     limit_train_batches=0.,
    # limit_val_batches=0.0,
    num_nodes=1,
    # gpus=1,
    # accelerator="gpu",
)

Multiprocessing is handled by SLURM.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(val_check_interval=1)` was configured so validation will run after every batch.


In [23]:
trainer.fit(model)


  | Name             | Type             | Params
------------------------------------------------------
0 | audio_transforms | AudioCompose     | 0     
1 | model            | AuditoryCNN      | 69.3 M
2 | loss_fn          | CrossEntropyLoss | 0     
3 | train_acc        | Accuracy         | 0     
4 | valid_acc        | Accuracy         | 0     
5 | test_acc         | Accuracy         | 0     
6 | test_confusion   | Accuracy         | 0     
------------------------------------------------------
69.3 M    Trainable params
0         Non-trainable params
69.3 M    Total params
277.070   Total estimated model params size (MB)
SLURM auto-requeueing enabled. Setting signal handlers.


len training set = 1050320                                                 
Epoch 0:   0%|          | 2/9277476560 [00:10<13233311:16:00,  5.14s/it, loss=1.19e-07, v_num=3.05e+7]

2023-05-04 18:35:06.151430: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 0:   0%|          | 16/9277476560 [00:46<7507705:51:52,  2.91s/it, loss=1.19e-07, v_num=3.05e+7]